# DistilBART CNN Summarization — AWS Marketplace

**Model**: `sshleifer/distilbart-cnn-12-6`  
**Task**: Abstractive text summarization  
**Benchmark**: ROUGE-2 21.26 on CNN/DailyMail  
**Price**: $0.10 / hr  
**License**: Apache-2.0  
**Input limit**: 1024 tokens

DistilBART is a distilled version of BART fine-tuned on the CNN/DailyMail dataset.
It generates fluent, abstractive summaries rather than extracting sentences verbatim.

**Prerequisites**
1. Subscribe to the product in AWS Marketplace and copy the Model Package ARN for your Region.
2. An IAM role with `AmazonSageMakerFullAccess`.
3. `pip install -U sagemaker boto3`

In [ ]:
# Paste the Model Package ARN from the AWS Marketplace listing
MODEL_PACKAGE_ARN = "<YOUR_MODEL_PACKAGE_ARN>"

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = session.boto_region_name

print(f"Region : {region}")
print(f"Role   : {role}")

## 1 — Deploy a real-time endpoint

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

ENDPOINT_NAME = "distilbart-cnn-summarization"

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.g4dn.xlarge",
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint: {ENDPOINT_NAME}")

## 2 — Single-document summarization

Send a JSON payload with `inputs` (the text to summarize), `max_length`,
and `min_length`. The model must receive ≤ 1024 tokens; longer documents
should be chunked or use the batch-transform path below.

In [ ]:
import json

runtime = boto3.client("sagemaker-runtime", region_name=region)

document = (
    "Amazon Web Services (AWS) is a subsidiary of Amazon that provides on-demand "
    "cloud computing platforms and APIs to individuals, companies, and governments, "
    "on a metered, pay-as-you-go basis. Clients will often use this in combination "
    "with autoscaling. AWS was launched in 2006 from the internal infrastructure that "
    "Amazon.com built in order to handle its online retail operations. AWS has expanded "
    "to provide over 200 services from data centers globally. Revenue generated by AWS "
    "in 2021 was $62.2 billion. This represented 13% of Amazon's total revenue."
)

payload = {
    "inputs": document,
    "max_length": 150,
    "min_length": 30,
}

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload),
)

result = json.loads(response["Body"].read())
print("Summary:")
print(result[0]["summary_text"])

## 3 — Batch Transform for large document corpora

For large-scale summarization (thousands of documents), use SageMaker Batch Transform.
Each line in the input JSONL file must be a self-contained JSON object with an `inputs` key.
Keep each document under 1024 tokens; split longer documents before upload.

**Input format** (`s3://your-bucket/input/documents.jsonl`):  
```
{"inputs": "Document one text...", "max_length": 150, "min_length": 30}
{"inputs": "Document two text...", "max_length": 150, "min_length": 30}
```

In [ ]:
import os

S3_BUCKET = session.default_bucket()
S3_INPUT_PREFIX  = "distilbart-cnn/input"
S3_OUTPUT_PREFIX = "distilbart-cnn/output"

# --- Write a local sample JSONL file and upload to S3 -----------------------
sample_docs = [
    {
        "inputs": (
            "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
            "It is named after the engineer Gustave Eiffel, whose company designed and built the tower "
            "from 1887 to 1889 as the centerpiece of the 1889 World's Fair. The tower is 330 metres tall "
            "and was the tallest man-made structure in the world for 41 years."
        ),
        "max_length": 80,
        "min_length": 20,
    },
    {
        "inputs": (
            "Machine learning is a method of data analysis that automates analytical model building. "
            "It is based on the idea that systems can learn from data, identify patterns and make "
            "decisions with minimal human intervention. Machine learning is a type of artificial "
            "intelligence that allows software applications to become more accurate at predicting "
            "outcomes without being explicitly programmed to do so."
        ),
        "max_length": 80,
        "min_length": 20,
    },
]

local_input = "/tmp/distilbart_input.jsonl"
with open(local_input, "w") as f:
    for doc in sample_docs:
        f.write(json.dumps(doc) + "\n")

s3_client = boto3.client("s3", region_name=region)
input_key  = f"{S3_INPUT_PREFIX}/documents.jsonl"
s3_client.upload_file(local_input, S3_BUCKET, input_key)
print(f"Uploaded input to s3://{S3_BUCKET}/{input_key}")

In [ ]:
from sagemaker.transformer import Transformer

# Re-use the already-registered model package
bt_model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

transformer = Transformer(
    model_name=bt_model.name if hasattr(bt_model, 'name') else "distilbart-cnn-bt",
    instance_count=1,
    instance_type="ml.g4dn.xlarge",
    output_path=f"s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}",
    content_type="application/json",
    accept="application/json",
    assemble_with="Line",
    sagemaker_session=session,
)

transformer.transform(
    data=f"s3://{S3_BUCKET}/{S3_INPUT_PREFIX}",
    data_type="S3Prefix",
    content_type="application/json",
    split_type="Line",
    wait=True,
)

print(f"Batch transform complete. Results at: s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}")

In [ ]:
# --- Download and display batch results -------------------------------------
output_key = f"{S3_OUTPUT_PREFIX}/documents.jsonl.out"
local_output = "/tmp/distilbart_output.jsonl"

s3_client.download_file(S3_BUCKET, output_key, local_output)

print("Batch summaries:")
with open(local_output) as f:
    for i, line in enumerate(f, 1):
        result = json.loads(line)
        if isinstance(result, list):
            summary = result[0].get("summary_text", "")
        else:
            summary = result.get("summary_text", "")
        print(f"[{i}] {summary}")

## 4 — Teardown

Delete the endpoint when finished to stop billing ($0.10/hr while running).

In [ ]:
predictor.delete_endpoint()
print(f"Endpoint '{ENDPOINT_NAME}' deleted.")